In [0]:
select * from gizmobox.bronze.vorders;

In [0]:
select value,
        regexp_replace(value, '"order_date":\\s*(\\d{4}-\\d{2}-\\d{2})' , '"order_date": "\$1"') AS fixed_value
        from gizmobox.bronze.vorders

In [0]:
create or replace temp view tvorders
as
select value,
        regexp_replace(value, '"order_date":\\s*(\\d{4}-\\d{2}-\\d{2})' , '"order_date": "\$1"') AS fixed_value
        from gizmobox.bronze.vorders

In [0]:
select schema_of_json(fixed_value) AS schema
from tvorders
LIMIT 1

In [0]:
select from_json(
    fixed_value,'
STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'
) AS json_value
from tvorders
LIMIT 1

In [0]:
create table if not exists gizmobox.silver.orders_json
as
select from_json(
    fixed_value,'
STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'
) AS json_value
from tvorders

In [0]:
select * from gizmobox.silver.orders_json